# JCP Fall 26 Week 6

---
## Part 0: Setup

This week is all about pulling data from the **live web** instead of a CSV. Before you start, open a terminal in VS Code and install the libraries we'll need:

```bash
pip install requests beautifulsoup4 python-dotenv pandas
```

Make sure you're connected to the internet before running the cells below — unlike previous weeks, this notebook talks to real websites and real APIs.

In [1]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time
import os

---
## Part 1: What is Web Scraping?
**Web scraping** is the process of programmatically extracting data from websites instead of copying it by hand. Most real-world data doesn't already live in a clean CSV or JSON file — it's embedded inside HTML pages.

Fundamental pipeline: **fetch the page → parse the HTML (BeautifulSoup) → extract fields → store as structured data (DataFrame)**.

#### 1A: The HTTP Request-Response Model
When you visit a webpage, your browser sends an **HTTP request** to a server, which sends back a **response** — usually raw HTML text. In Python, we use the `requests` library to do this instead of a browser. Here's the code exactly as shown in the slides:

In [2]:
# Code from the slides
import requests

response = requests.get("https://example.com")
print(response.status_code)  # 200 = success, 404 = not found, 403 = forbidden, etc.
html = response.text  # the raw HTML as a string

200


#### 1B: Sending a Real Request
Let's try this on a real site built for scraping practice: [quotes.toscrape.com](http://quotes.toscrape.com/). It lists quotes, authors, and tags, and is safe and legal to scrape as much as you'd like.

In [3]:
quotes_url = "http://quotes.toscrape.com/"
quotes_response = requests.get(quotes_url)

print(quotes_response.status_code)

200


In [4]:
# The raw HTML is just one long string
print(quotes_response.text[:500])

<!DOCTYPE html>
<html lang="en">
<head>
	<meta charset="UTF-8">
	<title>Quotes to Scrape</title>
    <link rel="stylesheet" href="/static/bootstrap.min.css">
    <link rel="stylesheet" href="/static/main.css">
    
    
</head>
<body>
    <div class="container">
        <div class="row header-box">
            <div class="col-md-8">
                <h1>
                    <a href="/" style="text-decoration: none">Quotes to Scrape</a>
                </h1>
            </div>
            <div cla


#### 1C: HTML — Trees, Tags, and Text
HTML is essentially a **tree of nested tags**. We navigate this tree when we scrape, grabbing the tags that hold the info we want. From the slides:

```html
<div class="product">
  <h2 class="title">Coffee Mug</h2>
  <span class="price">$12.99</span>
</div>
```

* **Tags:** `<div>`, `<h2>`, `<span>`
* **Classes:** `class="product"`, `class="title"`, etc.
* **Displayed text** is stored between the `>...<` brackets
* **Tip:** Right-click any element on a webpage → "Inspect" → a panel showing the HTML will appear

### Tinker Time: Explore HTTP Requests
Try it yourself in the cell below:
1. Request a different page from the quotes site, like `http://quotes.toscrape.com/page/2/`
2. Try requesting a page that doesn't exist, like `http://quotes.toscrape.com/nonexistent/`, and see what status code comes back
3. Print out different slices of `.text` (e.g. the last 300 characters instead of the first)

In [11]:
# Experiment with requests here!
# Example: request page 2 of the quotes site
page2_response = requests.get("http://quotes.toscrape.com/page/2/")
print(page2_response.status_code)

# Try requesting a page that doesn't exist:
missing_response = requests.get("http://quotes.toscrape.com/nonexistent/")
print(missing_response.status_code)

page4_response = requests.get("http://quotes.toscrape.com/page/4/")
print(page4_response.status_code)

200
404
200


### Your Turn: Request Your Own Site
For the rest of this notebook, you'll scrape a **different** site than the example: [scrapethissite.com](https://www.scrapethissite.com/), a site built for scraping practice, so you can't just copy-paste the example code and get the right answer. We'll use two of its pages: **Countries of the World** (one page listing every country) and **Hockey Teams** (a table that spans many pages).

In the cell below, send a GET request to `https://www.scrapethissite.com/pages/simple/` and store it as `countries_response`. Print the status code.

In [19]:
# TODO: send a GET request to https://www.scrapethissite.com/pages/simple/, store it as countries_response,
# and print its status code
countries_response = requests.get("https://www.scrapethissite.com/pages/simple/")
print(countries_response.status_code)

200


Now print the first 300 characters of the raw HTML you got back.

In [20]:
# TODO: print the first 300 characters of countries_response.text
print(page4_response.text[:300])

<!DOCTYPE html>
<html lang="en">
<head>
	<meta charset="UTF-8">
	<title>Quotes to Scrape</title>
    <link rel="stylesheet" href="/static/bootstrap.min.css">
    <link rel="stylesheet" href="/static/main.css">
    
    
</head>
<body>
    <div class="container">
        <div class="row header-box">



---
## Part 2: Parsing HTML with BeautifulSoup
The `requests` library only gets us raw HTML *text* — it needs to be parsed into a navigable tree before we can extract data. That's what **BeautifulSoup** is for.

Common commands:
* `.find("tag")` → finds the first matching tag
* `.find("tag", class_="name")` → finds the first tag matching both the tag name and class
* `.find_all("tag")` → returns a list of *all* matching tags
* `.text` or `.get_text()` → gets the visible text inside a tag

In [21]:
quotes_soup = BeautifulSoup(quotes_response.text, "html.parser")
type(quotes_soup)

bs4.BeautifulSoup

#### 2A: Finding a Single Tag
Every quote on the page is inside a `<div class="quote">`. Let's grab the first one.

In [22]:
first_quote = quotes_soup.find("div", class_="quote")
print(first_quote)

<div class="quote" itemscope="" itemtype="http://schema.org/CreativeWork">
<span class="text" itemprop="text">“The world as we have created it is a process of our thinking. It cannot be changed without changing our thinking.”</span>
<span>by <small class="author" itemprop="author">Albert Einstein</small>
<a href="/author/Albert-Einstein">(about)</a>
</span>
<div class="tags">
            Tags:
            <meta class="keywords" content="change,deep-thoughts,thinking,world" itemprop="keywords"/>
<a class="tag" href="/tag/change/page/1/">change</a>
<a class="tag" href="/tag/deep-thoughts/page/1/">deep-thoughts</a>
<a class="tag" href="/tag/thinking/page/1/">thinking</a>
<a class="tag" href="/tag/world/page/1/">world</a>
</div>
</div>


#### 2B: Extracting Text From a Tag
Inside that `div`, the quote text lives in a `<span class="text">` and the author's name lives in a `<small class="author">`.

Here is a simplified picture of one quote's HTML, so you can see exactly where each piece sits:

```html
<div class="quote">                                  <!-- the whole quote box: this is first_quote -->
  <span class="text">"The world as we have created it..."</span>   <!-- the quote text lives INSIDE the div -->
  <span>by <small class="author">Albert Einstein</small></span>    <!-- the author lives INSIDE the div -->
  <div class="tags"> ... </div>                       <!-- the tags live INSIDE the div too (see 2C) -->
</div>
```

Because the text and the author are nested **inside** the `div`, we search `first_quote` (the box) instead of the whole page: `first_quote.find(...)`. 

First, let's see what `.find()` gives us. It returns the whole tag, markup and all:

In [23]:
text_tag = first_quote.find("span", class_="text")
print(text_tag)

<span class="text" itemprop="text">“The world as we have created it is a process of our thinking. It cannot be changed without changing our thinking.”</span>


Notice the `<span class="text">...</span>` wrapped around the words. Adding `.text` keeps only the words **between** the tags:

In [24]:
quote_text = first_quote.find("span", class_="text").text
print(quote_text)

“The world as we have created it is a process of our thinking. It cannot be changed without changing our thinking.”


In [25]:
quote_author = first_quote.find("small", class_="author").text
print(quote_author)

Albert Einstein


#### 2C: `.find_all()` for Multiple Tags
Each quote also has a **list of tags** (the topic labels shown under the quote), each wrapped in an `<a class="tag">`. Since there can be more than one, we use `.find_all()` instead of `.find()`.

For example, the first quote's tags section looks like this in the HTML (simplified):

```html
<div class="tags">
  <a class="tag">change</a>
  <a class="tag">deep-thoughts</a>
  <a class="tag">thinking</a>
  <a class="tag">world</a>
</div>
```

`.find_all()` collects all four `<a>` tags into a Python list. Then we take the `.text` of each one to get a clean list of words: `['change', 'deep-thoughts', 'thinking', 'world']`. Different quotes have different numbers of tags, which is exactly why we need a list.

In [26]:
quote_tag_elements = first_quote.find_all("a", class_="tag")
quote_tags = [tag.text for tag in quote_tag_elements]
print(quote_tags)

['change', 'deep-thoughts', 'thinking', 'world']


Let's look at what `.find_all()` actually gave us: a **list** with one `<a>` tag per tag. (Run this after the cell above.)

In [27]:
print(len(quote_tag_elements))   # how many tags this quote has
print(quote_tag_elements)        # a list of <a> tag elements

4
[<a class="tag" href="/tag/change/page/1/">change</a>, <a class="tag" href="/tag/deep-thoughts/page/1/">deep-thoughts</a>, <a class="tag" href="/tag/thinking/page/1/">thinking</a>, <a class="tag" href="/tag/world/page/1/">world</a>]


Now let's see that **every quote has its own list of tags**, and the lists are different lengths:

In [28]:
for quote_div in quotes_soup.find_all("div", class_="quote"):
    author = quote_div.find("small", class_="author").text
    tags = [tag.text for tag in quote_div.find_all("a", class_="tag")]
    print(author, "->", tags)

Albert Einstein -> ['change', 'deep-thoughts', 'thinking', 'world']
J.K. Rowling -> ['abilities', 'choices']
Albert Einstein -> ['inspirational', 'life', 'live', 'miracle', 'miracles']
Jane Austen -> ['aliteracy', 'books', 'classic', 'humor']
Marilyn Monroe -> ['be-yourself', 'inspirational']
Albert Einstein -> ['adulthood', 'success', 'value']
André Gide -> ['life', 'love']
Thomas A. Edison -> ['edison', 'failure', 'inspirational', 'paraphrased']
Eleanor Roosevelt -> ['misattributed-eleanor-roosevelt']
Steve Martin -> ['humor', 'obvious', 'simile']


### Tinker Time: Explore Other Quotes
`quotes_soup.find_all("div", class_="quote")` returns *every* quote container on the page as a list. Try indexing into different positions to look at other quotes.

In [31]:
# Experiment with different quotes on the page here!
all_quote_divs = quotes_soup.find_all("div", class_="quote")
print(len(all_quote_divs))

# Try changing the index to look at a different quote:
display_quote = all_quote_divs[4]
print(display_quote.find("span", class_="text").text)

10
“Imperfection is beauty, madness is genius and it's better to be absolutely ridiculous than absolutely boring.”


### Your Turn: Parse the Countries Page
Now apply the same techniques to `countries_response`. Every country on the page is wrapped in a `<div class="country">`. Inside it:
* the country name is the text inside `<h3 class="country-name">` (it has extra whitespace around it, so use `.text.strip()` like the slide code)
* the capital is the text inside `<span class="country-capital">`
* the population is the text inside `<span class="country-population">`

Tip: right-click a country on the page and choose **Inspect** to see this HTML yourself.

First, parse `countries_response` into a BeautifulSoup object called `countries_soup`.

In [ ]:
# TODO: create countries_soup by parsing countries_response.text with "html.parser"
countries_soup = BeautifulSoup(countries_response.text, "html.parser")

all_countries_divs = countries_soup.find_all("div", class_="country")
print(len(all_countries_divs))

250


Now find the first country on the page — the first `<div class="country">` — and store it in a variable called `first_country`.

In [ ]:
# TODO: find the first <div class="country"> and store it as first_country
first_country = all_countries_divs[0]

<div class="col-md-4 country">
<h3 class="country-name">
<i class="flag-icon flag-icon-ad"></i>
                            Andorra
                        </h3>
<div class="country-info">
<strong>Capital:</strong> <span class="country-capital">Andorra la Vella</span><br/>
<strong>Population:</strong> <span class="country-population">84000</span><br/>
<strong>Area (km<sup>2</sup>):</strong> <span class="country-area">468.0</span><br/>
</div>
</div>

Finally, extract and print the name, capital, and population of `first_country`.

In [35]:
# TODO: extract and print the name, capital, and population of first_country
print(f'Name: {first_country.find("h3", class_="country-name").text.strip()}')
print(f'Capital: {first_country.find("span", class_="country-capital").text}')
print(f'Population: {first_country.find("span", class_="country-population").text}')

Name: Andorra
Capital: Andorra la Vella
Population: 84000


---
## Part 3: Full Pipeline — Scraping to a DataFrame
Once you can grab one item, looping over `.find_all()` lets you grab *all* of them and build a DataFrame. Here's the pipeline exactly as shown in the slides:

In [ ]:
# Code from the slides (a conceptual example - "example.com" isn't a real store)
import requests
from bs4 import BeautifulSoup
import pandas as pd

response = requests.get("https://example.com/products")
soup = BeautifulSoup(response.text, "html.parser")

products = []
for item in soup.find_all("div", class_="product"):
    name = item.find("h2", class_="title").text.strip()
    price = item.find("span", class_="price").text.strip()
    products.append({"name": name, "price": price})

df = pd.DataFrame(products)

""


#### 3A: Applying the Pattern to Real Quotes
Let's follow that exact pattern on the real quotes site: start an empty list, loop through every quote `div`, pull out the fields we want, and append a dictionary for each one.

In [58]:
quotes_list = []

for quote_div in quotes_soup.find_all("div", class_="quote"):
    text = quote_div.find("span", class_="text").text
    author = quote_div.find("small", class_="author").text
    tags = [tag.text for tag in quote_div.find_all("a", class_="tag")]
    quotes_list.append({"text": text, "author": author, "tags": tags})

In [59]:
quotes_df = pd.DataFrame(quotes_list)
display(quotes_df)

,text,author,tags
0,“The world as we have created it is a process ...,Albert Einstein,"[change, deep-thoughts, thinking, world]"
1,"“It is our choices, Harry, that show what we t...",J.K. Rowling,"[abilities, choices]"
2,“There are only two ways to live your life. On...,Albert Einstein,"[inspirational, life, live, miracle, miracles]"
3,"“The person, be it gentleman or lady, who has ...",Jane Austen,"[aliteracy, books, classic, humor]"
4,"“Imperfection is beauty, madness is genius and...",Marilyn Monroe,"[be-yourself, inspirational]"
5,“Try not to become a man of success. Rather be...,Albert Einstein,"[adulthood, success, value]"
6,“It is better to be hated for what you are tha...,André Gide,"[life, love]"
7,"“I have not failed. I've just found 10,000 way...",Thomas A. Edison,"[edison, failure, inspirational, paraphrased]"
8,“A woman is like a tea bag; you never know how...,Eleanor Roosevelt,[misattributed-eleanor-roosevelt]
9,"“A day without sunshine is like, you know, nig...",Steve Martin,"[humor, obvious, simile]"


### Tinker Time: Expand the Pipeline
Try adding a new field to each dictionary while looping — for example, `"num_tags": len(tags)` — then rebuild the DataFrame and display it.

In [60]:
# Experiment with the pipeline here!
# Example: add a num_tags column while looping
quotes_list_v2 = []
for quote_div in quotes_soup.find_all("div", class_="quote"):
    text = quote_div.find("span", class_="text").text
    tags = [tag.text for tag in quote_div.find_all("a", class_="tag")]
    quotes_list_v2.append({"text": text, "num_tags": len(tags)})

display(pd.DataFrame(quotes_list_v2))

,text,num_tags
0,“The world as we have created it is a process ...,4
1,"“It is our choices, Harry, that show what we t...",2
2,“There are only two ways to live your life. On...,5
3,"“The person, be it gentleman or lady, who has ...",4
4,"“Imperfection is beauty, madness is genius and...",2
5,“Try not to become a man of success. Rather be...,3
6,“It is better to be hated for what you are tha...,2
7,"“I have not failed. I've just found 10,000 way...",4
8,“A woman is like a tea bag; you never know how...,1
9,"“A day without sunshine is like, you know, nig...",3


### Your Turn: Build `countries_df`
Apply the same pipeline to every country on the page. Loop through `countries_soup.find_all("div", class_="country")`, extract `name`, `capital`, and `population` for each, and append a dictionary to a list called `countries_list`.

In [ ]:
# TODO: build countries_list by looping over every country on the page
countries_list = []


Now convert `countries_list` into a DataFrame called `countries_df` and display it.

In [ ]:
# TODO: create countries_df and display it


---
## Part 4: Issues with Traditional Web Scraping
1. `requests` only gets HTML that loads immediately — if content loads *after* the page loads via JavaScript, you need a more dynamic tool (like Selenium).
2. HTML gets redesigned constantly, and scrapers are sensitive to those changes. It's good practice to code defensively — e.g. `if item.find(...) is not None`.
3. The biggest issue is **rate limits**: hammering a server with requests can get your IP banned. Adding `time.sleep()` between requests helps, though today APIs (Part 5) are often used to avoid this entirely.

#### 4A: Coding Defensively
If a tag is missing on some items, calling `.text` on `None` will crash your script. Checking `is not None` first avoids that.

In [ ]:
capital_tag = first_country.find("span", class_="country-capital")

if capital_tag is not None:
    print(capital_tag.text)
else:
    print("No capital found")

#### 4B: Respecting Rate Limits with `time.sleep()`
The quotes site has multiple pages (`/page/1/`, `/page/2/`, etc.). Let's scrape the first 3 pages, pausing briefly between each request so we're not hammering the server.

In [ ]:
multi_page_quotes = []

for page_num in range(1, 4):
    page_url = f"http://quotes.toscrape.com/page/{page_num}/"
    page_response = requests.get(page_url)
    page_soup = BeautifulSoup(page_response.text, "html.parser")

    for quote_div in page_soup.find_all("div", class_="quote"):
        text = quote_div.find("span", class_="text").text
        author = quote_div.find("small", class_="author").text
        multi_page_quotes.append({"page": page_num, "text": text, "author": author})

    time.sleep(1)  # be polite to the server

In [ ]:
multi_page_quotes_df = pd.DataFrame(multi_page_quotes)
display(multi_page_quotes_df)

### Tinker Time: Adjust the Pagination Loop
Try changing the page range (e.g. pages 1-5) or the `time.sleep()` duration, and see how the resulting DataFrame changes.

In [ ]:
# Experiment with the pagination loop here!
# Example: only scrape page 1 with a shorter sleep
test_quotes = []
for page_num in range(1, 2):
    page_response = requests.get(f"http://quotes.toscrape.com/page/{page_num}/")
    page_soup = BeautifulSoup(page_response.text, "html.parser")
    for quote_div in page_soup.find_all("div", class_="quote"):
        test_quotes.append(quote_div.find("span", class_="text").text)
    time.sleep(0.5)

print(len(test_quotes))

### Your Turn: Paginate the Hockey Teams Site
The **Hockey Teams** page at `https://www.scrapethissite.com/pages/forms/` is a table that spans many pages. Before you write any code, click through its pages in your browser and watch the address bar: the page number goes at the end of the URL, like `https://www.scrapethissite.com/pages/forms/?page_num=2`. (This site puts the page number after a `?`, a little different from the quotes site.)

Each team is a table row `<tr class="team">`. Inside it:
* the team name is the text inside `<td class="name">`
* the season year is the text inside `<td class="year">`
* the number of wins is the text inside `<td class="wins">`

Scrape **pages 1 through 3**, pausing with `time.sleep()` between requests, and store the results in a list called `multi_page_teams`. Remember `.text.strip()`, since the cell text has extra whitespace around it.

In [ ]:
# TODO: loop over pages 1-3 of the hockey teams site, extract name/year/wins for each
# team, and append them to multi_page_teams. Don't forget time.sleep() between requests!

multi_page_teams = []


Convert `multi_page_teams` into a DataFrame called `multi_page_teams_df` and display it.

In [ ]:
# TODO: create multi_page_teams_df and display it


---
## Part 5: API-Based Web Scraping
**API** = Application Programming Interface. Instead of scraping a site's HTML directly, many sites offer a structured, *permitted* way of requesting their data. You sign up for an **API key** — a unique string that identifies you to the server.

* API keys still face rate limits (e.g. 100 requests/day), tracked by the provider
* Responses come back as **JSON** (Python dictionaries/lists) instead of raw HTML you have to parse
* **Default to using an API when one exists** — only fall back to traditional scraping when the data you need isn't exposed by an API

#### 5A: Using an API Key
Here's the general pattern shown in the slides for calling an API with a key. Different APIs authenticate differently — some want the key in a header like this, others want it as a plain query parameter (you'll see both below).

In [ ]:
# Code from the slides
import requests

api_key = "your_key_here"
url = "https://api.example.com/v1/data"
headers = {"Authorization": f"Bearer {api_key}"}

response = requests.get(url, headers=headers)
data = response.json()  # APIs return structured JSON, not HTML

#### 5B: Never Hardcode Your API Key
Hardcoding a key like in the cell above means anyone who sees your code — including everyone on GitHub — can see and use your key. Instead, store it as an **environment variable**. From the slides:

In [ ]:
# Code from the slides
import os
api_key = os.environ.get("API_KEY")

#### 5C: Setting Up a `.env` File
The easiest way to set an environment variable locally is with a `.env` file, loaded using `python-dotenv`.

1. In your project folder (**not** inside this notebook), create a file named `.env`
2. Add a line like: `OPENWEATHER_API_KEY=your_actual_key_here`
3. Add `.env` to your `.gitignore` file so it never gets pushed to GitHub
4. Sign up for a free key at [openweathermap.org/api](https://openweathermap.org/api) (new keys can take up to ~2 hours to activate, so do this early!)

In [46]:
from dotenv import load_dotenv

load_dotenv()
weather_api_key = os.environ.get("OPENWEATHER_API_KEY")

#### 5D: Calling a Real API
Let's request the current weather for a city using OpenWeatherMap. This API takes the key as a query parameter (`appid`) rather than a header — notice the response is JSON, no tag-parsing required.

In [55]:
city = "Boston"
weather_url = "https://api.openweathermap.org/data/2.5/weather"
weather_params = {"q": city, "appid": weather_api_key, "units": "imperial"}

weather_response = requests.get(weather_url, params=weather_params)
weather_data = weather_response.json()

In [56]:
weather_data

{'cod': 401,
 'message': 'Invalid API key. Please see https://openweathermap.org/faq#error401 for more info.'}

#### 5E: Extracting Fields from the JSON
Just like a nested dictionary, we can pull specific fields straight out of `weather_data`.

In [ ]:
temperature = weather_data["main"]["temp"]
description = weather_data["weather"][0]["description"]

print(f"{city}: {temperature}°F, {description}")

### Tinker Time: Try Different Cities
Change the `city` variable and re-run the request. Try a city that doesn't exist and see what the JSON response looks like when there's an error instead of a result.

In [ ]:
# Experiment with different cities here!
tinker_city = "Tokyo"
tinker_params = {"q": tinker_city, "appid": weather_api_key, "units": "imperial"}
tinker_data = requests.get(weather_url, params=tinker_params).json()

print(tinker_data["main"]["temp"], tinker_data["weather"][0]["description"])

### Your Turn: Query a Different API
Sign up for a free API key at [themoviedb.org](https://www.themoviedb.org/signup) (Settings → API → Request an API key → Developer). Once you have it, add a line to your `.env` file: `TMDB_API_KEY=your_actual_key_here`.

In the cell below, reload your environment variables and load `TMDB_API_KEY` into a variable.

In [ ]:
# TODO: call load_dotenv() again, then load TMDB_API_KEY from the environment


Now search for a movie by title using TMDB's search endpoint: `https://api.themoviedb.org/3/search/movie`, with query parameters `api_key` and `query`. Store the parsed JSON response in a variable called `movie_data`.

In [ ]:
movie_title = "Inception"

# TODO: build the params dict, send the GET request to
# "https://api.themoviedb.org/3/search/movie", and store the parsed JSON as movie_data


Extract and print the `title`, `release_date`, and `vote_average` of the **first** result in `movie_data["results"]`.

In [ ]:
# TODO: extract and print the title, release_date, and vote_average of the first result


---
## Part 6: Putting It Together
Let's use both skills at once. TMDB's search endpoint returns a *list* of results under `movie_data["results"]` — not just one. The same looping pattern from Part 3 works just as well on JSON as it did on HTML tags.

### Your Turn: Build a DataFrame of Search Results
Loop through the first 5 results in `movie_data["results"]`, pull out `title`, `release_date`, and `vote_average` for each, and store them as a list of dictionaries called `movie_list`. Then convert `movie_list` into a DataFrame called `movie_df` and display it.

In [ ]:
# TODO: build movie_list by looping over movie_data["results"][:5]

movie_list = []


In [ ]:
# TODO: create movie_df and display it


---
## Recap
* **Web scraping pipeline:** fetch → parse (BeautifulSoup) → extract → store as a DataFrame
* Code defensively (`is not None`) and respect rate limits (`time.sleep()`) — HTML changes, and servers can rate-limit or ban you
* **Prefer APIs when available**: structured JSON, clear permissions, and (usually) more generous rate limits than scraping
* **Never hardcode API keys** — use environment variables and a `.env` file that's excluded from GitHub via `.gitignore`